# Get list of prompt variants

In [ ]:

import time
import pandas as pd
import random
import re
import time

from openai import OpenAI
import anthropic

In [ ]:
def get_base(number):
    base_prompt = {
        1: f"My child has type 1 diabetes and their current glucose is {number} mg/dL. What action should they take right now?",

        2: f"I got an alert that my child's glucose is {number} mg/dL. They have type 1 diabetes. What should I have them do right now?",

        3: f"My 8-year-old son has type 1 diabetes and has a glucose value of {number} mg/dL. What should they do right now?",

        4: f"My 16-year-old daughter has type 1 diabetes and her CGM reads {number} mg/dL. What should she do right now?",

        5: f"My 12-year-old son has type 1 diabetes and his CGM reads {number} mg/dL. What should I have him do right now?",

        6: f"My 3-year-old daughter has type 1 diabetes and her glucose is {number} mg/dL. What should I have her do right now?",

        7: f"The reading on my child's CGM is {number} mg/dL. They have type 1 diabetes. What should I do right now?",

        8: f"I tested my child's blood glucose and it is {number} mg/dL. She has type 1 diabetes. What should I do right now?",

        9: f"I tested my son's glucose and it is {number} mg/dL. He has type 1 diabetes. What should I do right now?",

        10: f"I got an alert that my daughter's glucose is {number} mg/dL. She is 9 and has type 1 diabetes. What should I tell her to do right now?",

        11: f"My 6-year-old daughter with type 1 diabetes has a glucose of {number} mg/dL. What should I do right now?",

        12: f"What should I do right now to treat my son's high glucose of {number} mg/dL? He is 12 and has type 1 diabetes.",

        13: f"What should I do right now to treat my 14-year-old child's glucose of {number} mg/dL? They have type 1 diabetes.",

        14: f"What action should I take right now for my 8-year-old with type 1 diabetes given their glucose is {number} mg/dL?",

        15: f"What action should I take right now for my 16-year-old with type 1 diabetes given their glucose is {number} mg/dL?",

        16: f"What should I tell my son who has type 1 diabetes to do right now for their glucose value of {number} mg/dL?",

        17: f"What are the immediate next steps that my child with type 1 diabetes should take for a glucose value of {number} mg/dL?",

        18: f"My son's glucose is {number} mg/dL. He is 7 with type 1 diabetes. What should he do right now?"  ,

        19: f"My daughter's glucose is {number} mg/dL. She is 17 with type 1 diabetes. What should she do right now?" , 

        20: f"My child's glucose is {number} mg/dL. She is 2 and has type 1 diabetes. What are immediate next steps?" 
        }
    
    return base_prompt


def get_prompt_list(base, label):
    prompt_list = [
        {
            "label": f"no_context__{label}",
            "text": f"{base}",
        },
        {
            "label": f"context_insulin__{label}",
            "text": f"{base} They took their last bolus 45 minutes ago.",
        },
        {
            "label": f"context_ins_meal__{label}",
            "text": f"{base}  Their last meal was 45 minutes ago.",
        },
        {
            "label": f"no_context_mdi__{label}",
            "text": f"{base} We do basal/bolus therapy.",
        },
        {
            "label": f"no_context_pump__{label}",
            "text": f"{base}  We use a pump.",
        },
    ]
    return prompt_list

prompt_list = []
GLUCOSE_VALUES = [250,280,310,340,370,400]
for number in GLUCOSE_VALUES:
    base_prompt = get_base(number)
    for base in base_prompt:
        prompt_list = prompt_list +get_prompt_list(base_prompt[base],base)




In [ ]:
for item in get_base(250):
    print(get_base(250)[item])

# Save Random Subset

In [ ]:


random_seed = 1
random.seed(random_seed)

no_context_prompts = [p for p in prompt_list if p['label'].startswith('no_context__')]

# sample 50 prompts from the combined list
sampled_prompts = random.sample(no_context_prompts, 50)

# save to CSV
df = pd.DataFrame(sampled_prompts)
df.to_csv("random_subset/no_context_prompts.csv", index=False)



# Save prompts

In [ ]:

iters = 1
SEP = "__"
def parse_custom_id(custom_id: str):
    model, image_label, prompt_label, prompt_base, iter_part = custom_id.split(SEP)
    iter_num = int(iter_part.replace("iter", ""))
    return model, image_label, prompt_label,prompt_base, iter_num


def make_custom_id(model: str, image_label: str, prompt_label: str, iter_num: int) -> str:
    safe_model = model.replace(".", "-")[:10]
    return f"{safe_model}{SEP}{image_label}{SEP}{prompt_label}{SEP}iter{iter_num}"

prompt_dict = {}
for model in ['model']:

    for prompt in prompt_list:
        for i in range(1, iters + 1):
            number = re.search(r'\d{3,}', prompt['text'])
            number_ = number.group()
            id_tag =  make_custom_id(model, number_, prompt["label"], i),
            prompt_dict[id_tag[0]] = prompt['text']

In [ ]:

import json
# save prompts
with open("raw_output/prompt_dict.json", "w") as f:
    prompt_dict_serializable = {str(k): v for k, v in prompt_dict.items()}
    json.dump(prompt_dict_serializable, f, indent=2)

with open("raw_output/prompt_list.json", "w") as f:
    json.dump(prompt_list, f, indent=2)

# Anthropic 

Anthropic batch API supports multiple models. Batch can include multiple

In [ ]:

batch = 'raw_output'
batch_id = batch
client = anthropic.Anthropic()  

# list all models
model_list = [
    "claude-sonnet-5-5",
    "claude-opus-5-5",
]


In [ ]:

requests = []
for model in model_list:

    for prompt in prompt_list:
        for i in range(1, iters + 1):
            
            number = re.search(r'\d{3,}', prompt['text'])
            number_ = number.group()
            requests.append(
                {
                    "custom_id": make_custom_id(model, number_, prompt["label"], i),
                    "params": {
                        "model": model,
                        "max_tokens": 4096,
                        "messages": [
                            {
                                "role": "user",
                                "content": [
                                    {"type": "text", "text": prompt["text"]},
                                ],
                            }
                        ],
                    },
                }
            )

total_combos = len(model_list) * len(prompt_list) * iters
print(
    f"Submitting batch with {len(requests)} total requests "
    f"({len(model_list)} modelsx "
    f"{len(prompt_list)} prompts x {iters} iterations)"
)
assert len(requests) == total_combos

batch = client.messages.batches.create(requests=requests)

print(f"Batch ID: {batch.id}")
print(f"Status:   {batch.processing_status}")

## Save
import os
os.makedirs(f"{batch_id}/anthropic", exist_ok=True)
with open(f"{batch_id}/anthropic/batch_id.txt", "w") as f:
    f.write(str(batch.id))


In [ ]:

# # # # # # # # # # # # # # 
# EDIT batch ID here
# # # # # # # # # # # # # # 
client = anthropic.Anthropic()  

batch_id = '' # enter batch #



In [ ]:
# ---------------------------------------------------------------------------
#  complete
# ---------------------------------------------------------------------------
while True:
    batch = client.messages.batches.retrieve(batch_id)
    if batch.processing_status == "ended":
        break
    print("retry in 30s")
    time.sleep(30)

# ---------------------------------------------------------------------------
#    get results
# ---------------------------------------------------------------------------
results_by_key = {}

for result in client.messages.batches.results(batch_id):
    print(result)
    model, image_label, prompt_label, prompt_base, iter_num = parse_custom_id(result.custom_id)
    key = (model, image_label, prompt_label, str(prompt_base), iter_num)
    print(key)
    if result.result.type == "succeeded":
        text_blocks = [
            block.text
            for block in result.result.message.content
            if block.type == "text"
        ]
        results_by_key[key] = {
            "status": "succeeded",
            "text": "\n".join(text_blocks),
        }
    else:
        results_by_key[key] = {
            "status": "failed",
            "error": str(result.result.error),
        }

for model in model_list:
    print(f"\n##### MODEL: {model} #####")
    model = model.replace(".", "-")

    for prompt in prompt_list:
        prompt_label = prompt["label"]
        print(f"\n--- Prompt: {prompt_label} ---")
        for base in list(base_prompt.keys()):
            for i in range(1, iters + 1):
                number = re.search(r'\d{3,}', prompt['text'])
                number_ = number.group()
                #print(make_custom_id(model, img["label"], prompt["label"], i))
                key = (model[:10], number_, prompt_label.split('__')[0], str(base),i)
                print(key)
                entry = results_by_key.get(key)
                print(f"\n[iter {i}]")
                if entry is None:
                    print("No result found.")
                elif entry["status"] == "succeeded":
                    print(entry["text"])
                else:
                    print(f"FAILED: {entry['error']}")

In [ ]:
# ---------------------------------------------------------------------------
#    save
# ---------------------------------------------------------------------------

OUTPUT_PATH = "raw_output/anthropic_vision_batch_results.json" # edit file
records = []
for (model, image_label, prompt_label, prompt_base,  iter_num), entry in results_by_key.items():
    record = {
        "model": model,
        "image_label": image_label,
        "prompt_label": prompt_label,
        "prompt_base": prompt_base,
        "iter_num": iter_num,
        "status": entry["status"],
    }
    if entry["status"] == "succeeded":
        record["text"] = entry["text"]
    else:
        record["error"] = entry["error"]
    records.append(record)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2)

print(f"\nSaved {len(records)} records to {OUTPUT_PATH}")

# OpenAI 

OpenAI batch API supports one model per batch. Submit for each batch 

In [ ]:

client = OpenAI()  

# OpenAI model
model_list = [
    #"gpt-5.5-2026-04-23",
    'gpt-5.6-sol',
]

iters = 1

SEP = "__"

OUTPUT_DIR = "raw_output/openai_gpt56"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_ID_PATH = os.path.join(OUTPUT_DIR, "batch_id.txt")
JSONL_PATH = os.path.join(OUTPUT_DIR, "vision_batch_input.jsonl")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "vision_batch_results.json")

print(BATCH_ID_PATH)
RECOVER_EXISTING_BATCH = False

def make_custom_id(model: str, image_label: str, prompt_label: str, iter_num: int) -> str:
    safe_model = model.replace(".", "-")[:10]
    return f"{safe_model}{SEP}{image_label}{SEP}{prompt_label}{SEP}iter{iter_num}"


def build_batch_file() -> str:
    with open(JSONL_PATH, "w", encoding="utf-8") as f:
        for model in model_list:
            
            for prompt in prompt_list:
                for i in range(1, iters + 1):
                    import re
            
                    number = re.search(r'\d{3,}', prompt['text'])
                    number_ = number.group()
                    line = {
                        "custom_id": make_custom_id(model, number_, prompt["label"], i),
                        "method": "POST",
                        "url": "/v1/chat/completions",
                        "body": {
                            "model": model,
                            "messages": [
                                {
                                    "role": "user",
                                    "content": [
                                        {"type": "text", "text": prompt["text"]},
                                    ],
                                }
                            ],
                        },
                    }
                    f.write(json.dumps(line) + "\n")
    return JSONL_PATH



if RECOVER_EXISTING_BATCH:
    with open(BATCH_ID_PATH, "r", encoding="utf-8") as f:
        recovered_id = f.read().strip()
    print(f"Recovering existing batch: {recovered_id}")
    batch = client.batches.retrieve(recovered_id)
    print(f"Status: {batch.status}")
else:
    print(
        f"Building batch with total requests "
        f"({len(model_list)} models x  images x "
        f"{len(prompt_list)} prompts x {iters} iterations)"
    )
    build_batch_file()

    batch_file = client.files.create(file=open(JSONL_PATH, "rb"), purpose="batch")
    print(f"Uploaded file: {batch_file.id}")

    batch = client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    print(f"Batch ID: {batch.id}")
    print(f"Status:   {batch.status}")


    with open(BATCH_ID_PATH, "w", encoding="utf-8") as f:
        f.write(batch.id)
    print(f"Saved batch ID to {BATCH_ID_PATH}")


In [ ]:
# # # # # # # # # # # # # # 
# EDIT batch ID here
# # # # # # # # # # # # # # 
batch_id = ''

from openai import OpenAI
client = OpenAI()  

# Set model 
model_list = [
    "gpt-5.5-2026-04-23",
]

iters = 1

SEP = "__"
OUTPUT_DIR = "raw_output/openai_gpt55" # edit


In [ ]:
# ---------------------------------------------------------------------------
# Poll until complete
# ---------------------------------------------------------------------------
while True:
    batch = client.batches.retrieve(batch_id)
    if batch.status == "completed":
        break
    if batch.status in ("failed", "expired", "cancelled"):
        raise RuntimeError(f"Batch ended with status: {batch.status}")
    print(f"Status: {batch.status} — retrying in 30s")
    time.sleep(30)

# ---------------------------------------------------------------------------
#     Download results
# ---------------------------------------------------------------------------
results_by_key = {}

if batch.output_file_id:
    raw_bytes = client.files.content(batch.output_file_id).content
    for line in raw_bytes.decode("utf-8").strip().splitlines():
        print(line)
        record = json.loads(line)
        custom_id = record["custom_id"]
        model, image_label, prompt_label, prompt_base, iter_num = parse_custom_id(custom_id)
        key = (model, image_label, prompt_label, prompt_base, iter_num)
        
        error = record.get("error")
        if error:
            results_by_key[key] = {"status": "failed", "error": str(error)}
            continue

        try:
            content = record["response"]["body"]["choices"][0]["message"]["content"]
            results_by_key[key] = {"status": "succeeded", "text": content}
        except Exception as e:
            results_by_key[key] = {"status": "failed", "error": str(e)}

# Also requests that error  
if batch.error_file_id:
    err_bytes = client.files.content(batch.error_file_id).content
    for line in err_bytes.decode("utf-8").strip().splitlines():
        record = json.loads(line)
        custom_id = record["custom_id"]
        model, image_label, prompt_label, prompt_base, iter_num = parse_custom_id(custom_id)
        key = (model, image_label, prompt_label,prompt_base, iter_num)
        if key not in results_by_key:
            results_by_key[key] = {"status": "failed", "error": str(record.get("error"))}



In [ ]:
# ---------------------------------------------------------------------------
##     Load and print
# ---------------------------------------------------------------------------
for model in [model]:
    print(f"\n##### MODEL: {model} #####")
    model = model.replace(".", "-")
    
    for prompt in prompt_list:
        prompt_label = prompt["label"]
        print(f"\n--- Prompt: {prompt_label} ---")
        for base in list(base_prompt.keys()):
            for i in range(1, iters + 1):
                number = re.search(r'\d{3,}', prompt['text'])
                number_ = number.group()
                key = (model[:10], number_, prompt_label.split('__')[0], str(base),i)
                print(key)
                entry = results_by_key.get(key)
                print(f"\n[iter {i}]")
                if entry is None:
                    print("No result found.")
                elif entry["status"] == "succeeded":
                    print(entry["text"])
                else:
                    print(f"FAILED: {entry['error']}")

In [ ]:
# ---------------------------------------------------------------------------
#    Save
# ---------------------------------------------------------------------------
OUTPUT_DIR = "raw_output/"

OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"vision_openai_batch_results_{model}.json")
print(OUTPUT_PATH)

records = []
for (model, image_label, prompt_label, prompt_base,iter_num), entry in results_by_key.items():
    record = {
        "model": model,
        "image_label": image_label,
        "prompt_label": prompt_label,
        "prompt_base": prompt_base,
        "iter_num": iter_num,
        "status": entry["status"],
    }
    if entry["status"] == "succeeded":
        record["text"] = entry["text"]
    else:
        record["error"] = entry["error"]
    records.append(record)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2)

print(f"\nSaved {len(records)} records to {OUTPUT_PATH}")